In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("scikit-learn/adult-census-income")
print(dataset)
# Save locally as CSV
dataset["train"].to_csv("adult_census_income_train.csv")

In [ ]:

from datasets import load_dataset

dataset = load_dataset("scikit-learn/adult-census-income", split="train")


In [ ]:
from datasets import load_dataset

dataset = load_dataset("scikit-learn/adult-census-income", split="train")

# Convert to Pandas DataFrame for easier manipulation
df = dataset.to_pandas()

# 2. Basic Inspection
print(f"Dataset Shape: {df.shape}")
print("-" * 30)
print("First 5 rows:")
display(df.head())

In [ ]:
dataset

In [ ]:
#converting to pandas dataframe for easier manipulation
df=dataset.to_pandas()
df

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
# 3. Data Types and Null Values
print("\nData Info:")
print(df.info())

# 4. Statistical Summary
print("\nNumerical Statistics:")
display(df.describe())

In [ ]:
df.drop_duplicates()

In [ ]:
sns.set(style="whitegrid")

# 1. Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='income', data=df, hue="income", palette='ocean')
plt.title('Target Distribution: Income')
plt.show()

In [ ]:
#numerical distribution (histogram)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Categorical Distribution
cat_cols = ['workclass', 'education', 'marital.status', 'relationship']

plt.figure(figsize=(15, 10))
for i, col in enumerate(cat_cols):
    plt.subplot(2, 2, i + 1)
    sns.countplot(y=col, data=df, order=df[col].value_counts().index, hue=col, palette='muted')
    plt.title(f'Count of {col}')
plt.tight_layout()
plt.show()

 Understanding IQR

** Outliers can skew statistical measures and ruin model performance (especially linear models).


** IQR: The difference between the 75th percentile (Q3) and the 25th percentile (Q1).

 ** Bounds: Lower = Q1 - 1.5IQR,
 Upper = Q3 + 1.5IQR.


Q1 = df['column'].quantile(0.25)

Q3 = df['column'].quantile(0.75)

In [ ]:
# Focus on numerical columns that looked skewed on Day 1
focus_cols = ['age', 'hours.per.week', 'fnlwgt']

plt.figure(figsize=(15, 5))
for i, col in enumerate(focus_cols):

    plt.subplot(1, 3, i + 1)
    sns.boxplot(x=df[col], color='orange')
    plt.title(f'Boxplot of {col}')
plt.show()

In [ ]:
def handle_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"Column: {column} | IQR: {IQR} | Lower: {lower_bound} | Upper: {upper_bound}")

    # Let's check how many outliers we have
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Number of outliers identified: {len(outliers)}")


    # Strategy: Capping (Winsorizing)
    # We cap values to the upper and lower bounds instead of deleting rows (to preserve data)

    df_clean = df.copy()
    df_clean[column] = np.where(df_clean[column] > upper_bound, upper_bound,
                                np.where(df_clean[column] < lower_bound, lower_bound, df_clean[column]))

    return df_clean

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(x=df['age'])
plt.title('Age After Outlier Capping')

plt.subplot(1, 3, 2)
sns.boxplot(x=df['hours.per.week'])
plt.title('Hours/Week After Outlier Capping')

plt.subplot(1, 3, 3)
sns.boxplot(x=df['fnlwgt'])
plt.title('Fnlwgt After Outlier Capping')

plt.tight_layout()
plt.show()


In [ ]:

df.describe()


# Feature Scaling / Normalization — Why It Matters

## Without Scaling
When features are on very different scales (e.g., age vs. income):

- ❌ **Slow or failed convergence**
- ❌ **Biased feature importance**
- ❌ **Poor model performance**


---

## With Scaling
After scaling features to a similar range:

- ✅ **Faster training**
- ✅ **Better accuracy**
- ✅ **More stable gradients**
- ✅ **Fair feature contribution**

**Why?**
All features contribute more evenly, helping gradient-based optimizers move efficiently toward the optimum.


## Simple Intuition
> **Normalization = “Put everything between 0 and 1”**

This prevents one feature from overpowering the others just because of its units.

---

## Common Scaling Methods

### 1. Min–Max Normalization
Scales values to \([0, 1]\)

$$
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$


Best when:
- You want a fixed range
- No extreme outliers

---

Best when:
- Data is roughly normal
- Using models like Logistic Regression, SVM, Linear Regression

---

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Isolate numerical columns to scale
cols_to_scale = ['age', 'fnlwgt', 'education.num', 'hours.per.week', 'capital.gain', 'capital.loss']

# 1. Standardization (Preferred for this dataset which has outliers/variance)
scaler_std = StandardScaler()

# Create a copy to keep original for comparison
df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler_std.fit_transform(df[cols_to_scale])

In [ ]:
df_scaled

In [ ]:
df.describe()

In [ ]:
df_scaled.describe()

In [ ]:
# 2. Check the effect
print("Original Mean (Age):", df['age'].mean())
print("Scaled Mean (Age):", round(df_scaled['age'].mean(), 2)) # Should be 0
print("Scaled Std (Age):", round(df_scaled['age'].std(), 2))   # Should be 1

In [ ]:
# 2. Check the effect
print("Original Mean (capital.gain):", df['capital.gain'].mean())
print("Scaled Mean (capital.gain):", round(df_scaled['capital.gain'].mean(), 2)) # Should be 0
print("Scaled Std (capital.gain):", round(df_scaled['capital.gain'].std(), 2))   # Should be 1

In [ ]:
# 3. Visualization: Before vs After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Before
sns.kdeplot(df['age'], ax=ax1, fill=True, color='r')
ax1.set_title('Before Scaling (Age)')

# After
sns.kdeplot(df_scaled['age'], ax=ax2, fill=True, color='b')
ax2.set_title('After Standardization (Age)')
plt.show()

In [ ]:
# 3. Visualization: Before vs After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Before
sns.kdeplot(df['capital.gain'], ax=ax1, fill=True, color='r')
ax1.set_title('Before Scaling (capital.gain)')

# After
sns.kdeplot(df_scaled['capital.gain'], ax=ax2, fill=True, color='b')
ax2.set_title('After Standardization (capital.gain)')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1. Label Encoding the Target Variable
le = LabelEncoder()
df['income_encoded'] = le.fit_transform(df['income'])

print("Target Encoding Mapping:")
for i, item in enumerate(le.classes_):
    print(f"{item} --> {i}")

# Drop original income column
df.drop('income', axis=1, inplace=True)

In [ ]:
df

In [ ]:
df.shape

In [ ]:
# 2. One-Hot Encoding for Nominal Features
# Nominal: Workclass, Marital Status, Occupation, Relationship, Race, Sex
nominal_cols = ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex']

# Using Pandas get_dummies (easiest for EDA phase)
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(f"Shape after One-Hot Encoding: {df_encoded.shape}")

In [ ]:
df_encoded

In [ ]:
df['race'].unique()

In [ ]:
df['sex'].value_counts()

In [ ]:
df['race'].value_counts()

In [ ]:
# 3. Label Encoding for Ordinal Features?
if 'education' in df_encoded.columns:
    df_encoded.drop('education', axis=1, inplace=True)

# Update main dataframe
df = df_encoded
display(df.head())

In [ ]:
if 'native.country' in df_encoded.columns:
    df_encoded.drop('native.country', axis=1, inplace=True)
df = df_encoded
display(df.head())

## Feature Engineering & Selection

### 🔧 Feature Engineering
- **Definition:** Creating new columns or features from existing data to better represent the underlying problem.
- **Purpose:** Helps models capture important patterns and relationships that may not be obvious in the raw data.
- **Example:**
  ```text
  Net Capital = capital.gain - capital.loss

In [ ]:
df['capital.gain'] = (
    df['capital.gain']
      .replace('?', np.nan)
      .ffill()
)

df['capital.loss'] = (
    df['capital.loss']
      .replace('?', np.nan)
      .ffill()
)

In [ ]:
df['capital.gain']=df['capital.gain'].ffill()
df['capital.loss']=df['capital.loss'].ffill()

In [ ]:
df['net_capital']=df['capital.gain']-df['capital.loss']
df

In [ ]:
df['net_capital'].head()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(50, 20))
# Calculate correlation only on numeric columns
corr_matrix = df.corr(numeric_only=True)

# Plot heatmap
sns.heatmap(corr_matrix, annot=True, cmap='rainbow', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout
plt.show()

In [ ]:
df.shape

In [ ]:
# 3. Feature Selection based on Correlation with Target
target_corr = corr_matrix['income_encoded'].sort_values(ascending=False)
print("\nTop 10 features correlated with Income (>50K):")
print(target_corr)



In [ ]:
# Drop columns with extremely low correlation (optional noise reduction)
# keeping features with abs(correlation) > 0.01
relevant_features = target_corr[abs(target_corr) > 0.01].index.tolist()
df_final = df[relevant_features]
print(f"\nReduced feature set from {df.shape[1]} to {df_final.shape[1]}")

In [ ]:
df_final

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate Features (X) and Target (y)
X = df_final.drop('income_encoded', axis=1)
y = df_final['income_encoded']

# 2. Split
# stratify=y ensures the proportion of >50k and <=50k is the same in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data Splitting Complete:")
print(f"Training Set: {X_train.shape}")
print(f"Test Set: {X_test.shape}")

# 3. Save processed data (Optional)
# X_train.to_csv('X_train_processed.csv', index=False)


In [ ]:
y_train.shape

In [ ]:
y_test.shape

In [ ]:
X_train.shape

In [ ]:
X_test.shape


In [ ]:
from sklearn.model_selection import train_test_split

X = [[1],[2],[3],[4],[5],[6],[7],[8],[9],[10]]
y = [0, 0, 1, 1, 1,0,1,0,1,1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print("X_train:", X_train)
print("X_test:", X_test)
print("y_train:", y_train)
print("y_test:", y_test)